# Qualitative Dataset Preparation

This notebook documents the path from raw FNSPID Nasdaq news data to the processed qualitative tables used by the multimodal model.

The raw Nasdaq file is too large to load directly with pandas, so the actual processing is handled by Python scripts in `processing/qualitative_prep/`. This notebook explains the steps, optionally runs the scripts, and previews each generated CSV.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

repo_root = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "README.md").exists() and (candidate / "processing").exists():
        repo_root = candidate
        break

qual_dir = repo_root / "processing/qualitative_prep"
raw_news_path = repo_root / "data/raw/nasdaq_exteral_data.csv"
trading_calendar_path = repo_root / "data/raw/trading_calendar.csv"
quant_dataset_path = repo_root / "final_dataset/quant_dataset_30day.csv"
output_dir = repo_root / "data/qualitative_dataset"

print(f"Repo root: {repo_root}")
print(f"Raw Nasdaq news exists: {raw_news_path.exists()} -> {raw_news_path}")
print(f"Trading calendar exists: {trading_calendar_path.exists()} -> {trading_calendar_path}")
print(f"Quantitative dataset exists: {quant_dataset_path.exists()} -> {quant_dataset_path}")
print(f"Qualitative output dir: {output_dir}")


def preview_csv(path, rows=5):
    path = Path(path)
    print(f"\n{path}")
    if not path.exists():
        print("Missing file. Run the pipeline step that creates it.")
        return None
    preview = pd.read_csv(path, nrows=rows)
    print(f"Columns: {list(preview.columns)}")
    display(preview)
    return preview

Repo root: c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye
Raw Nasdaq news exists: True -> c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\raw\nasdaq_exteral_data.csv
Trading calendar exists: True -> c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\raw\trading_calendar.csv
Quantitative dataset exists: True -> c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\quantitative_dataset\quant_dataset_30day.csv
Qualitative output dir: c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset


In [2]:
# Input overview: the quantitative dataset defines the ticker/prediction-date grid.
quantitative_df = pd.read_csv(quant_dataset_path, usecols=["ticker", "prediction_date"])
model_tickers = sorted(quantitative_df["ticker"].unique())

print(f"Model tickers: {len(model_tickers)}")
print(f"Ticker-date rows: {len(quantitative_df):,}")
print(f"Prediction date range: {quantitative_df['prediction_date'].min()} to {quantitative_df['prediction_date'].max()}")

pd.DataFrame({"ticker": model_tickers}).head(20)

Model tickers: 50
Ticker-date rows: 83,862
Prediction date range: 2020-02-13 to 2026-10-21


,ticker
0,AAPL
1,ABBV
2,ABT
3,ACN
4,ADBE
5,AMD
6,AMGN
7,AMZN
8,AVGO
9,AXP


In [3]:
# Optional: rerun the full qualitative processing pipeline.
# Leave RUN_PIPELINE as False when you only want to inspect existing outputs.
RUN_PIPELINE = False

pipeline_scripts = [
    "audit_ticker_mappings.py",
    "audit_nasdaq_coverage.py",
    "prepare_nasdaq_article_links.py",
    "build_company_day_news_windows.py",
]

if RUN_PIPELINE:
    for script in pipeline_scripts:
        script_path = qual_dir / script
        print(f"Running {script_path}...")
        subprocess.run([sys.executable, str(script_path)], check=True, cwd=repo_root)
else:
    print("Pipeline execution skipped. Set RUN_PIPELINE = True to regenerate outputs.")

Pipeline execution skipped. Set RUN_PIPELINE = True to regenerate outputs.


In [4]:
outputs = {
    "ticker_symbol_mapping_summary": output_dir / "ticker_symbol_mapping_summary.csv",
    "ticker_symbol_mapping_candidates": output_dir / "ticker_symbol_mapping_candidates.csv",
    "nasdaq_news_coverage_monthly_2020_2023": output_dir / "nasdaq_news_coverage_monthly_2020_2023.csv",
    "nasdaq_unique_articles_2020_2023": output_dir / "nasdaq_unique_articles_2020_2023.csv",
    "nasdaq_article_company_links_2020_2023": output_dir / "nasdaq_article_company_links_2020_2023.csv",
    "nasdaq_company_match_review_queue_2020_2023": output_dir / "nasdaq_company_match_review_queue_2020_2023.csv",
    "company_day_news_window_5session": output_dir / "company_day_news_window_5session.csv",
}

pd.DataFrame(
    [
        {
            "table": name,
            "path": path,
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / 1_000_000, 2) if path.exists() else None,
        }
        for name, path in outputs.items()
    ]
)

,table,path,exists,size_mb
0,ticker_symbol_mapping_summary,c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\ticker_symbol_mapping_summa...,True,0.00
1,ticker_symbol_mapping_candidates,c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\ticker_symbol_mapping_candi...,True,0.00
2,nasdaq_news_coverage_2020_2023,c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\nasdaq_news_coverage_2020_2...,True,0.01
3,nasdaq_unique_articles_2020_2023,c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\nasdaq_unique_articles_2020...,True,779.39
4,nasdaq_article_company_links_2020_2023,c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\nasdaq_article_company_link...,True,46.69
5,nasdaq_company_match_review_queue_2020_2023,c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\nasdaq_company_match_review...,True,18.32
6,company_day_news_window_5session,c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\company_day_news_window_5se...,True,15.56


## Step 1: Symbol Mapping Audit

The quantitative dataset and the Nasdaq news file do not always use the same symbol. Examples include `BRK-B` vs `BRK.B`, `META` vs historical `FB`, and `GOOGL` vs `GOOG`.

The mapping audit checks exact symbols and known alternates, then records which Nasdaq symbols should be counted for each modeling ticker.

In [5]:
preview_csv(outputs["ticker_symbol_mapping_candidates"], rows=10)


c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\ticker_symbol_mapping_candidates.csv
Columns: ['ticker', 'nasdaq_symbol', 'match_type', 'nasdaq_total_rows', 'rows_2017_2020', 'rows_2021', 'rows_2022_2023', 'present_in_nasdaq']


,ticker,nasdaq_symbol,match_type,nasdaq_total_rows,rows_2017_2020,rows_2021,rows_2022_2023,present_in_nasdaq
0,AAPL,AAPL,exact,9338,473,0,8865,True
1,ABBV,ABBV,exact,6451,2681,551,1777,True
2,ABT,ABT,exact,4055,1426,432,1347,True
3,ACN,ACN,exact,4387,1531,277,849,True
4,ADBE,ADBE,exact,6570,2375,417,1711,True
5,AMD,AMD,exact,9209,4269,812,4024,True
6,AMGN,AMGN,exact,5258,1931,330,1319,True
7,AMZN,AMZN,exact,5060,278,0,4782,True
8,AVGO,AVGO,exact,4156,3322,0,7,True
9,AXP,AXP,exact,3790,2122,0,7,True


,ticker,nasdaq_symbol,match_type,nasdaq_total_rows,rows_2017_2020,rows_2021,rows_2022_2023,present_in_nasdaq
0,AAPL,AAPL,exact,9338,473,0,8865,True
1,ABBV,ABBV,exact,6451,2681,551,1777,True
2,ABT,ABT,exact,4055,1426,432,1347,True
3,ACN,ACN,exact,4387,1531,277,849,True
4,ADBE,ADBE,exact,6570,2375,417,1711,True
5,AMD,AMD,exact,9209,4269,812,4024,True
6,AMGN,AMGN,exact,5258,1931,330,1319,True
7,AMZN,AMZN,exact,5060,278,0,4782,True
8,AVGO,AVGO,exact,4156,3322,0,7,True
9,AXP,AXP,exact,3790,2122,0,7,True


In [6]:
preview_csv(outputs["ticker_symbol_mapping_summary"], rows=10)


c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\ticker_symbol_mapping_summary.csv
Columns: ['ticker', 'matched_nasdaq_symbols', 'match_types', 'total_rows', 'rows_2017_2020', 'rows_2021', 'rows_2022_2023', 'has_any_nasdaq_symbol', 'needs_symbol_mapping']


,ticker,matched_nasdaq_symbols,match_types,total_rows,rows_2017_2020,rows_2021,rows_2022_2023,has_any_nasdaq_symbol,needs_symbol_mapping
0,AAPL,AAPL,exact,9338,473,0,8865,True,False
1,ABBV,ABBV,exact,6451,2681,551,1777,True,False
2,ABT,ABT,exact,4055,1426,432,1347,True,False
3,ACN,ACN,exact,4387,1531,277,849,True,False
4,ADBE,ADBE,exact,6570,2375,417,1711,True,False
5,AMD,AMD,exact,9209,4269,812,4024,True,False
6,AMGN,AMGN,exact,5258,1931,330,1319,True,False
7,AMZN,AMZN,exact,5060,278,0,4782,True,False
8,AVGO,AVGO,exact,4156,3322,0,7,True,False
9,AXP,AXP,exact,3790,2122,0,7,True,False


,ticker,matched_nasdaq_symbols,match_types,total_rows,rows_2017_2020,rows_2021,rows_2022_2023,has_any_nasdaq_symbol,needs_symbol_mapping
0,AAPL,AAPL,exact,9338,473,0,8865,True,False
1,ABBV,ABBV,exact,6451,2681,551,1777,True,False
2,ABT,ABT,exact,4055,1426,432,1347,True,False
3,ACN,ACN,exact,4387,1531,277,849,True,False
4,ADBE,ADBE,exact,6570,2375,417,1711,True,False
5,AMD,AMD,exact,9209,4269,812,4024,True,False
6,AMGN,AMGN,exact,5258,1931,330,1319,True,False
7,AMZN,AMZN,exact,5060,278,0,4782,True,False
8,AVGO,AVGO,exact,4156,3322,0,7,True,False
9,AXP,AXP,exact,3790,2122,0,7,True,False


## Step 2: Monthly Nasdaq Coverage Audit

Before building model inputs, we check whether FNSPID has source-tagged Nasdaq records for each ticker and month from 2020 through 2023.

This separates three cases:

- The source has records and full article text.
- The source has records but no article body.
- The source has no records for that ticker-month.

Month-level coverage is more useful than year-level coverage because a company can have news coverage in some months and none in others. `no news` is only meaningful inside a ticker-month where the source has usable coverage.

In [7]:
coverage_preview = preview_csv(outputs["nasdaq_news_coverage_monthly_2020_2023"], rows=12)

coverage = pd.read_csv(outputs["nasdaq_news_coverage_monthly_2020_2023"])
coverage_summary = coverage.groupby("coverage_month").agg(
    ticker_months=("ticker", "count"),
    ticker_months_with_source=("has_source_records", "sum"),
    ticker_months_with_full_text=("has_full_text_records", "sum"),
    source_records=("source_records", "sum"),
    full_text_records=("full_text_records", "sum"),
)
coverage_summary.head(12)


c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\nasdaq_news_coverage_2020_2023.csv
Columns: ['ticker', 'year', 'source_records', 'full_text_records', 'source_symbols_counted', 'has_source_records', 'has_full_text_records']


,ticker,year,source_records,full_text_records,source_symbols_counted,has_source_records,has_full_text_records
0,AAPL,2020,473,0,AAPL,True,False
1,AAPL,2021,0,0,NaN,False,False
2,AAPL,2022,3094,3094,AAPL,True,True
3,AAPL,2023,5771,5771,AAPL,True,True
4,ABBV,2020,773,495,ABBV,True,True
5,ABBV,2021,551,551,ABBV,True,True
6,ABBV,2022,809,809,ABBV,True,True
7,ABBV,2023,968,968,ABBV,True,True
8,ABT,2020,543,543,ABT,True,True
9,ABT,2021,432,432,ABT,True,True


,ticker_years,ticker_years_with_source,ticker_years_with_full_text,source_records,full_text_records
year,,,,,
2020,50,46,28,32021,19581
2021,50,29,29,20217,20217
2022,50,32,32,38000,38000
2023,50,38,38,60989,60989


## Step 3: Deduplicate Articles and Verify Company Links

The raw Nasdaq file can contain the same story under multiple source tags. We separate this into two tables:

- **Articles:** one row per unique story with stable `article_id`, publication data, headline/body, source tags, and first eligible trading session.
- **Article-company links:** one row per decision that a story concerns a modeling ticker.

A link is accepted when the headline/body contains a strong ticker reference or company-name evidence. Links that only have the original source tag but no text evidence are placed in the review queue.

In [8]:
preview_csv(outputs["nasdaq_unique_articles_2020_2023"], rows=5)


c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\nasdaq_unique_articles_2020_2023.csv
Columns: ['article_id', 'Date', 'eligible_trading_date', 'eligibility_rule', 'Article_title', 'Url', 'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary', 'original_source_tags']


,article_id,Date,eligible_trading_date,eligibility_rule,Article_title,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary,original_source_tags
0,00002b0d7b1f5025,2022-02-17 00:00:00 UTC,2022-02-18,midnight_or_date_only_next_session,"Don't Buy Disney Stock for Streaming, Buy It for This",https://www.nasdaq.com/articles/dont-buy-disney-stock-for-streaming-buy-it-for-this,NaN,NaN,Walt Disney (NYSE: DIS) turned in exactly the quarter investors need to see to send the stock higher after the recen...,"With high demand for a premium digital service, Disney should continue to see margins improve at the parks segment c...","Digital services are driving record profits for Disney's parks Revenue from the Parks, Experiences, and Products seg...","Digital services are driving record profits for Disney's parks Revenue from the Parks, Experiences, and Products seg...","Digital services are driving record profits for Disney's parks Revenue from the Parks, Experiences, and Products seg...",DIS
1,0001e3eaa45411c1,2022-12-29 00:00:00 UTC,2022-12-30,midnight_or_date_only_next_session,Better Dividend Stock for 2023? Intel Stock vs. Pepsi Stock,https://www.nasdaq.com/articles/better-dividend-stock-for-2023-intel-stock-vs.-pepsi-stock,NaN,NaN,"Intel (NASDAQ: INTC) and Pepsi (NASDAQ: PEP) might operate in different industries, but each pays a healthy dividend...","Intel (NASDAQ: INTC) and Pepsi (NASDAQ: PEP) might operate in different industries, but each pays a healthy dividend...","Intel (NASDAQ: INTC) and Pepsi (NASDAQ: PEP) might operate in different industries, but each pays a healthy dividend...","Intel (NASDAQ: INTC) and Pepsi (NASDAQ: PEP) might operate in different industries, but each pays a healthy dividend...","Intel (NASDAQ: INTC) and Pepsi (NASDAQ: PEP) might operate in different industries, but each pays a healthy dividend...",PEP
2,00020c71737a6be0,2022-05-02 00:00:00 UTC,2022-05-03,midnight_or_date_only_next_session,Wells Fargo is Now Oversold (WFC),https://www.nasdaq.com/articles/wells-fargo-is-now-oversold-wfc,NaN,NaN,"Legendary investor Warren Buffett advises to be fearful when others are greedy, and be greedy when others are fearfu...","In trading on Monday, shares of Wells Fargo & Co (Symbol: WFC) entered into oversold territory, hitting an RSI readi...",A bullish investor could look at WFC's 29.8 RSI reading today as a sign that the recent heavy selling is in the proc...,"In trading on Monday, shares of Wells Fargo & Co (Symbol: WFC) entered into oversold territory, hitting an RSI readi...","In trading on Monday, shares of Wells Fargo & Co (Symbol: WFC) entered into oversold territory, hitting an RSI readi...",WFC
3,000248b47ee6dd07,2021-12-31 00:00:00 UTC,2022-01-03,midnight_or_date_only_next_session,Can Disney Stock Go From Worst of the Dow to Best in 2022?,https://www.nasdaq.com/articles/can-disney-stock-go-from-worst-of-the-dow-to-best-in-2022,NaN,NaN,It was generally a good year for the 30 blue chips resident in the Dow Jones Industrial Average. A healthy 22 of 30 ...,"And yes, it's Walt Disney (NYSE: DIS). Disney stock is down 14% in 2021, double the slide of the Dow's second-bigges...",Disney+ played a starring role in the media giant's healthy gains in 2020. See the 10 stocks *Stock Advisor returns ...,Things don't have to go perfectly for Disney to come out on top; it's more than the biggest Dow sinker in 2021. Busi...,Things don't have to go perfectly for Disney to come out on top; it's more than the biggest Dow sinker in 2021. Busi...,DIS
4,00027db5c0528e49,2023-09-08 00:00:00 UTC,2023-09-11,midnight_or_date_only_next_session,Costco's Sales Update: Can Wins in Stores Offset Weakness Online?,https://www.nasdaq.com/articles/costcos-sales-update%3A-can-wins-in-stores-offset-weakness-online,NaN,NaN,Costco (NASDAQ: COST) stock has been a steady performer for investors this year. Shares have stayed ahead of the mar...,Shares have stayed ahead of the market rally so far in 2023 while beati

,article_id,Date,eligible_trading_date,eligibility_rule,Article_title,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary,original_source_tags
0,00002b0d7b1f5025,2022-02-17 00:00:00 UTC,2022-02-18,midnight_or_date_only_next_session,"Don't Buy Disney Stock for Streaming, Buy It for This",https://www.nasdaq.com/articles/dont-buy-disney-stock-for-streaming-buy-it-for-this,NaN,NaN,Walt Disney (NYSE: DIS) turned in exactly the quarter investors need to see to send the stock higher after the recen...,"With high demand for a premium digital service, Disney should continue to see margins improve at the parks segment c...","Digital services are driving record profits for Disney's parks Revenue from the Parks, Experiences, and Products seg...","Digital services are driving record profits for Disney's parks Revenue from the Parks, Experiences, and Products seg...","Digital services are driving record profits for Disney's parks Revenue from the Parks, Experiences, and Products seg...",DIS
1,0001e3eaa45411c1,2022-12-29 00:00:00 UTC,2022-12-30,midnight_or_date_only_next_session,Better Dividend Stock for 2023? Intel Stock vs. Pepsi Stock,https://www.nasdaq.com/articles/better-dividend-stock-for-2023-intel-stock-vs.-pepsi-stock,NaN,NaN,"Intel (NASDAQ: INTC) and Pepsi (NASDAQ: PEP) might operate in different industries, but each pays a healthy dividend...","Intel (NASDAQ: INTC) and Pepsi (NASDAQ: PEP) might operate in different industries, but each pays a healthy dividend...","Intel (NASDAQ: INTC) and Pepsi (NASDAQ: PEP) might operate in different industries, but each pays a healthy dividend...","Intel (NASDAQ: INTC) and Pepsi (NASDAQ: PEP) might operate in different industries, but each pays a healthy dividend...","Intel (NASDAQ: INTC) and Pepsi (NASDAQ: PEP) might operate in different industries, but each pays a healthy dividend...",PEP
2,00020c71737a6be0,2022-05-02 00:00:00 UTC,2022-05-03,midnight_or_date_only_next_session,Wells Fargo is Now Oversold (WFC),https://www.nasdaq.com/articles/wells-fargo-is-now-oversold-wfc,NaN,NaN,"Legendary investor Warren Buffett advises to be fearful when others are greedy, and be greedy when others are fearfu...","In trading on Monday, shares of Wells Fargo & Co (Symbol: WFC) entered into oversold territory, hitting an RSI readi...",A bullish investor could look at WFC's 29.8 RSI reading today as a sign that the recent heavy selling is in the proc...,"In trading on Monday, shares of Wells Fargo & Co (Symbol: WFC) entered into oversold territory, hitting an RSI readi...","In trading on Monday, shares of Wells Fargo & Co (Symbol: WFC) entered into oversold territory, hitting an RSI readi...",WFC
3,000248b47ee6dd07,2021-12-31 00:00:00 UTC,2022-01-03,midnight_or_date_only_next_session,Can Disney Stock Go From Worst of the Dow to Best in 2022?,https://www.nasdaq.com/articles/can-disney-stock-go-from-worst-of-the-dow-to-best-in-2022,NaN,NaN,It was generally a good year for the 30 blue chips resident in the Dow Jones Industrial Average. A healthy 22 of 30 ...,"And yes, it's Walt Disney (NYSE: DIS). Disney stock is down 14% in 2021, double the slide of the Dow's second-bigges...",Disney+ played a starring role in the media giant's healthy gains in 2020. See the 10 stocks *Stock Advisor returns ...,Things don't have to go perfectly for Disney to come out on top; it's more than the biggest Dow sinker in 2021. Busi...,Things don't have to go perfectly for Disney to come out on top; it's more than the biggest Dow sinker in 2021. Busi...,DIS
4,00027db5c0528e49,2023-09-08 00:00:00 UTC,2023-09-11,midnight_or_date_only_next_session,Costco's Sales Update: Can Wins in Stores Offset Weakness Online?,https://www.nasdaq.com/articles/costcos-sales-update%3A-can-wins-in-stores-offset-weakness-online,NaN,NaN,Costco (NASDAQ: COST) stock has been a steady performer for investors this year. Shares have stayed ahead of the mar...,Shares have stayed ahead of the market rally so far in 2023 while beati

In [9]:
links_preview = preview_csv(outputs["nasdaq_article_company_links_2020_2023"], rows=10)

links = pd.read_csv(outputs["nasdaq_article_company_links_2020_2023"], usecols=["match_status", "match_reason", "review_queue"])
links["match_status"].value_counts()


c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\nasdaq_article_company_links_2020_2023.csv
Columns: ['article_id', 'ticker', 'original_source_tag', 'source_tag_match_type', 'match_status', 'match_reason', 'review_queue', 'Date', 'eligible_trading_date', 'eligibility_rule', 'Article_title', 'Url']


,article_id,ticker,original_source_tag,source_tag_match_type,match_status,match_reason,review_queue,Date,eligible_trading_date,eligibility_rule,Article_title,Url
0,9e71392669365b01,GS,GS,exact,accepted,company_name_mention_in_body,False,2023-12-16 21:00:00 UTC,2023-12-18,non_trading_day_next_session,Apollo-owned Aspen Insurance files for US IPO,https://www.nasdaq.com/articles/apollo-owned-aspen-insurance-files-for-us-ipo
1,bd1ebfb728c1b38d,GS,GS,exact,accepted,strong_ticker_mention_in_body,False,2023-12-16 18:00:00 UTC,2023-12-18,non_trading_day_next_session,"DIA, GS, MCD, AMGN: ETF Inflow Alert",https://www.nasdaq.com/articles/dia-gs-mcd-amgn%3A-etf-inflow-alert
2,df9b5b2f6afa04b7,GS,GS,exact,accepted,strong_ticker_mention_in_headline,False,2023-12-16 15:00:00 UTC,2023-12-18,non_trading_day_next_session,Goldman (GS) Eyes Expansion of Credit Business in India,https://www.nasdaq.com/articles/goldman-gs-eyes-expansion-of-credit-business-in-india
3,9ebbd1ba14a0c4c6,GS,GS,exact,accepted,strong_ticker_mention_in_body,False,2023-12-16 14:00:00 UTC,2023-12-18,non_trading_day_next_session,GS Factor-Based Stock Analysis,https://www.nasdaq.com/articles/gs-factor-based-stock-analysis-11
4,85317d7c6bc7c47d,GS,GS,exact,accepted,strong_ticker_mention_in_body,False,2023-12-16 12:00:00 UTC,2023-12-18,non_trading_day_next_session,Can Dogs of the Dow ETFs Snap Losing Trend in 2024?,https://www.nasdaq.com/articles/can-dogs-of-the-dow-etfs-snap-losing-trend-in-2024
5,c03c4db9d47840a1,GS,GS,exact,accepted,strong_ticker_mention_in_body,False,2023-12-16 00:00:00 UTC,2023-12-18,midnight_or_date_only_next_session,BlackRock (BLK) Sued for Allegedly Misleading ESG Strategy,https://www.nasdaq.com/articles/blackrock-blk-sued-for-allegedly-misleading-esg-strategy
6,a68177ceeff333f3,GS,GS,exact,accepted,company_name_mention_in_body,False,2023-12-16 00:00:00 UTC,2023-12-18,midnight_or_date_only_next_session,Software startup ServiceTitan prepares for 2024 IPO-sources,https://www.nasdaq.com/articles/software-startup-servicetitan-prepares-for-2024-ipo-sources
7,1429a57ba4019e49,GS,GS,exact,accepted,company_name_mention_in_body,False,2023-12-16 00:00:00 UTC,2023-12-18,midnight_or_date_only_next_session,Sterling climbs as investors eye further gains in 2024,https://www.nasdaq.com/articles/sterling-climbs-as-investors-eye-further-gains-in-2024
8,4b6c2f067f852b41,GS,GS,exact,accepted,company_name_mention_in_body,False,2023-12-16 00:00:00 UTC,2023-12-18,midnight_or_date_only_next_session,"Shippers mask positions, weigh options amid Red Sea attacks",https://www.nasdaq.com/articles/shippers-mask-positions-weigh-options-amid-red-sea-attacks
9,7397656b422c7010,GS,GS,exact,accepted,company_name_mention_in_body,False,2023-12-16 00:00:00 UTC,2023-12-18,midnight_or_date_only_next_session,"Disruptions in Red Sea unlikely to have large effects on oil, LNG - Goldman",https://www.nasdaq.com/articles/disruptions-in-red-sea-unlikely-to-have-large-effects-on-oil-lng-goldman


match_status
accepted     139951
uncertain     11248
Name: count, dtype: int64

In [10]:
preview_csv(outputs["nasdaq_company_match_review_queue_2020_2023"], rows=10)


c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\nasdaq_company_match_review_queue_2020_2023.csv
Columns: ['article_id', 'ticker', 'original_source_tag', 'source_tag_match_type', 'match_status', 'match_reason', 'review_queue', 'Date', 'eligible_trading_date', 'eligibility_rule', 'Article_title', 'Url', 'Article', 'original_source_tags']


,article_id,ticker,original_source_tag,source_tag_match_type,match_status,match_reason,review_queue,Date,eligible_trading_date,eligibility_rule,Article_title,Url,Article,original_source_tags
0,04f4506b08ef138a,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-12-01 00:00:00 UTC,2023-12-04,midnight_or_date_only_next_session,"GLOBAL MARKETS-Shares gain, dollar slips as Fed's Powell sounds caution",https://www.nasdaq.com/articles/global-markets-shares-gain-dollar-slips-as-feds-powell-sounds-caution,"By Sinéad Carew and Amanda Cooper\nNEW YORK/LONDON, Dec 1 (Reuters) - MSCI's global stock index gained ground on Fri...",GS
1,b2f0d7d626969597,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-11-29 00:00:00 UTC,2023-11-30,midnight_or_date_only_next_session,"Up 24% Since The Beginning Of 2023, What Should You Expect From HSBC Stock?",https://www.nasdaq.com/articles/up-24-since-the-beginning-of-2023-what-should-you-expect-from-hsbc-stock,HSBC’s stock (NYSE: HSBC) has gained 24% YTD as compared to the 19% rise in the S&P500 index over the same period. F...,GS
2,940eeb822f09a085,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-11-22 00:00:00 UTC,2023-11-24,midnight_or_date_only_next_session,UK's Johnson Matthey raises outlook as energy focus grows,https://www.nasdaq.com/articles/uks-johnson-matthey-raises-outlook-as-energy-focus-grows,"By Eva Mathews\nNov 22 (Reuters) - Johnson Matthey JMAT.L, which makes catalytic converters and pollution filters fo...",GS
3,abffaf08e6ba1bdd,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-11-22 00:00:00 UTC,2023-11-24,midnight_or_date_only_next_session,"Up 10% In The Last One Month, What's Next For Morgan Stanley Stock?",https://www.nasdaq.com/articles/up-10-in-the-last-one-month-whats-next-for-morgan-stanley-stock,"Morgan Stanley’s stock (NYSE: MS) has lost roughly 6% YTD, as compared to the 18% rise in the S&P500 over the same p...",GS
4,4d88f97fea7c300f,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-11-08 00:00:00 UTC,2023-11-09,midnight_or_date_only_next_session,EXPLAINER-How tycoon and Chrysler Building owner Benko lost grip of his empire,https://www.nasdaq.com/articles/explainer-how-tycoon-and-chrysler-building-owner-benko-lost-grip-of-his-empire,"By Matthias Inverardi, Alexandra Schwarz-Goerlich and Tom\nSims\n VIENNA/DUESSELDORF/FRANKFURT, Nov 8 (Reu...",GS
5,0406446fc45bb1ae,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-10-15 00:00:00 UTC,2023-10-16,midnight_or_date_only_next_session,Bank of America Stock Has An 83% Upside To Its Pre-Inflation Shock,https://www.nasdaq.com/articles/bank-of-america-stock-has-an-83-upside-to-its-pre-inflation-shock,"Bank of America stock (NYSE: BAC) currently trades at $27 per share, around 45% below (83% upside) its level of $49 ...",GS
6,97798fc2da7f07a4,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-10-13 00:00:00 UTC,2023-10-16,midnight_or_date_only_next_session,GRAPHIC-How U.S. bank balance sheets have changed as Fed raises rates,https://www.nasdaq.com/articles/graphic-how-u.s.-bank-balance-sheets-have-changed-as-fed-raises-rates,Oct 13 (Reuters) - The Federal Reserve's aggressive monetary policy tightening is prompting U.S. banks to slow the f...,"GS,WFC"
7,4201ade66a7e55fa,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-10-11 00:00:00 UTC,2023-10-12,midnight_or_date_only_next_session,Brink's says not in deal talks with NCR,https://www.nasdaq.com/articles/brinks-says-not-in-deal-talks-with-ncr,"By Svea Herbst-Bayliss\nNEW YORK, Oct 11 (Reuters) - Cash management firm Brink's Co BCO.N said on Wednesday that it...",GS
8,509edf6b01b71162,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-10-11 00:00:00 UTC,2023-10-12,midnight_or_date_only_next_session,What To Expect From Citigroup Stock In Q3?,https://www.nasdaq.com/articles/what-to-expect-from-citigroup-stock-in-q3,"Citigroup (NYSE: C) is scheduled to report its fiscal Q3 2023 results on Fr

,article_id,ticker,original_source_tag,source_tag_match_type,match_status,match_reason,review_queue,Date,eligible_trading_date,eligibility_rule,Article_title,Url,Article,original_source_tags
0,04f4506b08ef138a,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-12-01 00:00:00 UTC,2023-12-04,midnight_or_date_only_next_session,"GLOBAL MARKETS-Shares gain, dollar slips as Fed's Powell sounds caution",https://www.nasdaq.com/articles/global-markets-shares-gain-dollar-slips-as-feds-powell-sounds-caution,"By Sinéad Carew and Amanda Cooper\nNEW YORK/LONDON, Dec 1 (Reuters) - MSCI's global stock index gained ground on Fri...",GS
1,b2f0d7d626969597,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-11-29 00:00:00 UTC,2023-11-30,midnight_or_date_only_next_session,"Up 24% Since The Beginning Of 2023, What Should You Expect From HSBC Stock?",https://www.nasdaq.com/articles/up-24-since-the-beginning-of-2023-what-should-you-expect-from-hsbc-stock,HSBC’s stock (NYSE: HSBC) has gained 24% YTD as compared to the 19% rise in the S&P500 index over the same period. F...,GS
2,940eeb822f09a085,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-11-22 00:00:00 UTC,2023-11-24,midnight_or_date_only_next_session,UK's Johnson Matthey raises outlook as energy focus grows,https://www.nasdaq.com/articles/uks-johnson-matthey-raises-outlook-as-energy-focus-grows,"By Eva Mathews\nNov 22 (Reuters) - Johnson Matthey JMAT.L, which makes catalytic converters and pollution filters fo...",GS
3,abffaf08e6ba1bdd,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-11-22 00:00:00 UTC,2023-11-24,midnight_or_date_only_next_session,"Up 10% In The Last One Month, What's Next For Morgan Stanley Stock?",https://www.nasdaq.com/articles/up-10-in-the-last-one-month-whats-next-for-morgan-stanley-stock,"Morgan Stanley’s stock (NYSE: MS) has lost roughly 6% YTD, as compared to the 18% rise in the S&P500 over the same p...",GS
4,4d88f97fea7c300f,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-11-08 00:00:00 UTC,2023-11-09,midnight_or_date_only_next_session,EXPLAINER-How tycoon and Chrysler Building owner Benko lost grip of his empire,https://www.nasdaq.com/articles/explainer-how-tycoon-and-chrysler-building-owner-benko-lost-grip-of-his-empire,"By Matthias Inverardi, Alexandra Schwarz-Goerlich and Tom\nSims\n VIENNA/DUESSELDORF/FRANKFURT, Nov 8 (Reu...",GS
5,0406446fc45bb1ae,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-10-15 00:00:00 UTC,2023-10-16,midnight_or_date_only_next_session,Bank of America Stock Has An 83% Upside To Its Pre-Inflation Shock,https://www.nasdaq.com/articles/bank-of-america-stock-has-an-83-upside-to-its-pre-inflation-shock,"Bank of America stock (NYSE: BAC) currently trades at $27 per share, around 45% below (83% upside) its level of $49 ...",GS
6,97798fc2da7f07a4,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-10-13 00:00:00 UTC,2023-10-16,midnight_or_date_only_next_session,GRAPHIC-How U.S. bank balance sheets have changed as Fed raises rates,https://www.nasdaq.com/articles/graphic-how-u.s.-bank-balance-sheets-have-changed-as-fed-raises-rates,Oct 13 (Reuters) - The Federal Reserve's aggressive monetary policy tightening is prompting U.S. banks to slow the f...,"GS,WFC"
7,4201ade66a7e55fa,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-10-11 00:00:00 UTC,2023-10-12,midnight_or_date_only_next_session,Brink's says not in deal talks with NCR,https://www.nasdaq.com/articles/brinks-says-not-in-deal-talks-with-ncr,"By Svea Herbst-Bayliss\nNEW YORK, Oct 11 (Reuters) - Cash management firm Brink's Co BCO.N said on Wednesday that it...",GS
8,509edf6b01b71162,GS,GS,exact,uncertain,source_tag_only_no_text_match,True,2023-10-11 00:00:00 UTC,2023-10-12,midnight_or_date_only_next_session,What To Expect From Citigroup Stock In Q3?,https://www.nasdaq.com/articles/what-to-expect-from-citigroup-stock-in-q3,"Citigroup (NYSE: C) is scheduled to report its fiscal Q3 2023 results on Fr

## Step 4: First Eligible Trading Session

Each article gets a first eligible trading session using `data/raw/trading_calendar.csv`.

Rules:

- A trustworthy timestamp before or at that session's actual market close can count that day.
- After-close articles count on the next trading session.
- Non-trading-day articles count on the next trading session.
- Date-only or suspicious midnight timestamps count on the next trading session after the stated date.

The result appears in the article and link tables as `eligible_trading_date` and `eligibility_rule`.

In [11]:
article_rules = pd.read_csv(
    outputs["nasdaq_unique_articles_2020_2023"],
    usecols=["eligibility_rule"],
)
article_rules["eligibility_rule"].value_counts()

eligibility_rule
midnight_or_date_only_next_session    117301
published_before_or_at_close             279
non_trading_day_next_session             191
after_close_next_session                   6
Name: count, dtype: int64

## Step 5: Company-Day News Windows

The final qualitative join table has one row per ticker and prediction date from the quantitative dataset.

For prediction date `t`, the news window is `t` plus the four preceding trading sessions. A story is included only when:

- its article-company link is accepted, and
- its `eligible_trading_date` falls inside that five-session window.

The table keeps every qualifying `article_id`. No cap is applied yet. It also records whether FNSPID Nasdaq source coverage is available for the date. Rows in 2024-2026 are retained but flagged out of the main multimodal comparison because the FNSPID Nasdaq coverage used here is only 2020-2023.

In [12]:
company_day_preview = preview_csv(outputs["company_day_news_window_5session"], rows=10)

company_day = pd.read_csv(
    outputs["company_day_news_window_5session"],
    usecols=[
        "ticker",
        "prediction_date",
        "news_count",
        "source_coverage_available",
        "no_eligible_news",
        "include_in_multimodal_comparison",
    ],
)

company_day_summary = pd.Series(
    {
        "rows": len(company_day),
        "rows_with_source_coverage": int(company_day["source_coverage_available"].sum()),
        "rows_excluded_from_multimodal_comparison": int((~company_day["include_in_multimodal_comparison"]).sum()),
        "rows_with_news": int((company_day["news_count"] > 0).sum()),
        "rows_with_no_eligible_news_inside_coverage": int(company_day["no_eligible_news"].sum()),
        "max_news_count": int(company_day["news_count"].max()),
    }
)
company_day_summary


c:\Users\eshaq\Documents\F26\Advanced Machine Learning\BullsEye\data\qualitative_dataset\company_day_news_window_5session.csv
Columns: ['ticker', 'prediction_date', 'article_ids', 'article_count', 'news_count', 'source_coverage_available', 'no_eligible_news', 'include_in_multimodal_comparison', 'no_accepted_news', 'news_window_sessions']


,ticker,prediction_date,article_ids,article_count,news_count,source_coverage_available,no_eligible_news,include_in_multimodal_comparison,no_accepted_news,news_window_sessions
0,AAPL,2020-02-13,NaN,0,0,True,True,True,True,5
1,ABBV,2020-02-13,"04f1690704292f0d,09cf80e3ce54189e,0d076a05731f73d9,22b817e057a65519,27823feb3117ed9c,2a8f25fe6526819d,2f1483e4f2ec9d...",28,28,True,False,True,False,5
2,ABT,2020-02-13,a64a2557f2e7f57d,1,1,True,False,True,False,5
3,ACN,2020-02-13,"2d25b7b506d34057,71af43a2db49d651,88d83d3de9635196,9bfc9adf8c31ce7b,acd1a0074ee8bf32,d4f1b832c8f46343,ef211727a88575bf",7,7,True,False,True,False,5
4,ADBE,2020-02-13,"4a28d9f6e4bc5038,772ce632629e14e9,86d942cad3c561cf,ae22f32aa8ee14eb",4,4,True,False,True,False,5
5,AMD,2020-02-13,"073e63386c81c306,0d0d1bc7138a0ea2,32b60b4d13a6daa4,3c7f6f5cabe5c8d3,6ef88af3182703bc,7b490a1de030c8a7,7d84b79fc2e1c6...",16,16,True,False,True,False,5
6,AMGN,2020-02-13,"0c225b9980f7b3e3,1dc3406c5a145264,29445b49274a9187,858fec08f9323d01,9b1b61bf61e78874",5,5,True,False,True,False,5
7,AMZN,2020-02-13,NaN,0,0,True,True,True,True,5
8,AVGO,2020-02-13,"3ed3b1ce753ec08f,4e5b4fb0d6984569,9da728a93dbe5fb4,b32320d9b06e2144,b8e3f1dc73e1adc1,c18ff9a783546597,c9393fdef1a0f6...",9,9,True,False,True,False,5
9,AXP,2020-02-13,"5f629b9ccc1e40bb,e900c68cabfc5d54",2,2,True,False,True,False,5


rows                                          83862
rows_with_source_coverage                     48662
rows_excluded_from_multimodal_comparison      35200
rows_with_news                                30411
rows_with_no_eligible_news_inside_coverage    18256
max_news_count                                  325
dtype: int64